# FlashEats — Class 5 Investigation

## Client escalation

> **“Late deliveries are increasing and customers say our ETA is unreliable. Figure out what is happening before we invest in an AI delay-prediction system.”**

Today is not a syntax lesson. Your job is to use SQL, CSV, JSON and APIs like an FDE:

**define the question → retrieve evidence → validate completeness → reconcile sources → state what you know and what you still cannot know**

### Rules
- Do not train an ML model.
- State your definition of **late** before calculating it.
- Preserve raw API responses.
- Do not silently drop failures.
- If another team gets a different answer, investigate the definition and data grain first.

In [1]:
# 0. INSTALLS + IMPORTS
!pip -q install pandas requests flask matplotlib

import os, sys, json, time, sqlite3, subprocess, zipfile
from pathlib import Path

import pandas as pd
import requests
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

print("Environment ready.")

Environment ready.


In [2]:
# 1. LOAD THE CLASSROOM PACK
# In Colab, upload FlashEats_Classroom_Pack.zip when prompted.

BASE = Path(r"d:\Code\JabTakNaaMaaneTinkuJiya\flasheats-classroom-pack")

if not BASE.exists():
    try:
        from google.colab import files
        print("Upload FlashEats_Classroom_Pack.zip")
        uploaded = files.upload()
        zip_name = next(name for name in uploaded if name.endswith(".zip"))
        with zipfile.ZipFile(zip_name, "r") as z:
            z.extractall("/content")
    except Exception as e:
        print("If running locally, set BASE manually to the extracted pack.")
        print(e)

print("BASE:", BASE)
print("Exists:", BASE.exists())

BASE: d:\Code\JabTakNaaMaaneTinkuJiya\flasheats-classroom-pack
Exists: True


## 2. Source map first

Before analysis, predict where you would look for:

| Information | Likely source | Why? |
|---|---|---|
| Promised delivery time |  |  |
| Actual delivery time |  |  |
| Driver assignment |  |  |
| Customer complaint |  |  |
| Restaurant status |  |  |
| Driver movement/events |  |  |

**Discuss:** Which source is authoritative for each fact, and which is only contextual?

### Answer: Source Map Predictions
| Information | Likely source | Why? |
|---|---|---|
| Promised delivery time | `orders` table | Established at checkout, central to the order record. |
| Actual delivery time | `orders` table / `driver_events` | Captured by the driver app when they swipe 'Delivered'. |
| Driver assignment | `drivers` / `orders` / `driver_events` | Dispatch system maps drivers to orders. |
| Customer complaint | Support tickets (CSV/Zendesk) | Customer service systems track these interactions. |
| Restaurant status | `restaurants` API/DB | Restaurant tablets report prep status. |
| Driver movement/events | `driver_events` JSON | Emitted via high-frequency GPS pings from driver app. |


In [3]:
# 3. BASIC SYNTAX — READ EVERY SOURCE

# SQLite
db_path = BASE / "database" / "flasheats.db"
con = sqlite3.connect(db_path)

tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
    con
)
display(tables)

sample_orders = pd.read_sql("SELECT * FROM orders LIMIT 5;", con)
display(sample_orders)

# CSV
tickets = pd.read_csv(BASE / "data" / "support_tickets.csv")
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv")
display(tickets.head())

# Nested JSON
with open(BASE / "data" / "driver_events.json", "r") as f:
    driver_events = json.load(f)

print("Driver objects:", len(driver_events))
display(driver_events[:1])



,name
0,customers
1,drivers
2,orders
3,restaurants


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear
2,O00003,C0855,R010,D039,Bengaluru,2026-08-01T19:35:00,2026-08-01T20:17:52.941967,2026-08-01T19:57:50.820366,2026-08-01T20:13:28.061191,delivered,2.42,medium,clear
3,O00004,C0229,R030,D038,Bengaluru,2026-08-09T18:16:00,2026-08-09T19:30:19.067006,2026-08-09T18:46:48.725485,NaN,cancelled,14.20,high,rain
4,O00005,C0040,R004,D047,Bengaluru,2026-08-06T20:35:00,2026-08-06T21:54:48,2026-08-06T21:18:38.024280,2026-08-06T22:15:49.290713,delivered,18.00,high,clear


,ticket_id,order_id,created_at,category,customer_message
0,T00001,NaN,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,NaN,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still not here.
2,T00003,NaN,2026-08-25T18:22:00,status_mismatch,The app says picking up but the restaurant says it is ready.
3,T00004,O00037,2026-08-09T21:09:00,Late Delivery,The rider has not moved for 15 minutes.
4,T00005,O00040,2026-08-11T20:15:00,late_delivery,My order is already past the promised time.


Driver objects: 120


[{'driver_id': 'D001',
  'events': [{'order_id': 'O00162',
    'type': 'assigned',
    'timestamp': '2026-08-26T17:53:25.977803'},
   {'order_id': 'O00162',
    'type': 'gps_ping',
    'timestamp': '2026-08-26T18:10:59.372420',
    'lat': 12.826339,
    'lon': 77.448282},
   {'order_id': 'O00162',
    'type': 'gps_ping',
    'timestamp': '2026-08-26T18:28:32.767037',
    'lat': 12.876013,
    'lon': 77.477707},
   {'order_id': 'O00162',
    'type': 'gps_ping',
    'timestamp': '2026-08-26T18:46:06.161653',
    'lat': 12.925686,
    'lon': 77.507133},
   {'order_id': 'O00162',
    'type': 'gps_ping',
    'timestamp': '2026-08-26T19:03:39.556270',
    'lat': 12.97536,
    'lon': 77.536558},
   {'order_id': 'O00162',
    'type': 'picked_up',
    'timestamp': '2026-08-26T18:27:32.131791'},
   {'order_id': 'O00162',
    'type': 'delivered',
    'timestamp': '2026-08-26T19:21:12.950887'},
   {'order_id': 'O00180',
    'type': 'assigned',
    'timestamp': '2026-08-08T18:05:08.439625'},
   {'o

# Challenge 1 — How large is the late-delivery problem?

Before coding, decide:

- **Late means:** ______________________________
- **Denominator:** _____________________________
- **Cancelled orders:** include / exclude / other?
- **Missing actual delivery timestamp:** what will you do?
- **Row grain:** what does one row represent?

### Required analysis
1. valid delivered orders,
2. late count and percentage,
3. median lateness among late orders,
4. worst delayed orders,
5. any data-quality issue that changes the metric.

In [4]:
# Starter syntax only — complete the analysis yourself.

# SQL examples:
# SELECT order_id, promised_eta, actual_delivery_at, final_status
# FROM orders
# WHERE ...

orders_df = pd.read_sql("SELECT * FROM orders", con)

query = """
SELECT *
FROM orders
LIMIT 20;
"""

orders_preview = pd.read_sql(query, con)
display(orders_preview)

# Useful pandas syntax:
# df["timestamp"] = pd.to_datetime(df["timestamp"], format="mixed", errors="coerce")
# df["delay_min"] = (df["actual"] - df["promised"]).dt.total_seconds() / 60
# df["order_id"].duplicated().sum()
# df.isna().sum()

# TODO: your Challenge 1 analysis

# 0. Deduplicate orders
order_analysis = orders_df.drop_duplicates("order_id", keep="first").copy()
total_orders = len(order_analysis)

# 1. Convert timestamp strings to pandas datetime objects
order_analysis["actual_delivery_at"] = pd.to_datetime(order_analysis["actual_delivery_at"], format="mixed", errors="coerce")
order_analysis["promised_eta"] = pd.to_datetime(order_analysis["promised_eta"], format="mixed", errors="coerce")

# 2. Filter down to ONLY delivered orders
delivered_df = order_analysis[order_analysis["final_status"] == "delivered"].copy()

# Calculate delay in minutes
delivered_df["delay_min"] = (
    delivered_df["actual_delivery_at"] - delivered_df["promised_eta"]
).dt.total_seconds() / 60

total_delivered = len(delivered_df)

# 3. Filter for orders that were late
late_orders_df = delivered_df[delivered_df["delay_min"] > 0]
num_late = len(late_orders_df)

# 4. Calculate percentage & median delay
percentage = (num_late / total_delivered) * 100
median_delay = late_orders_df["delay_min"].median()

# --- PRINT SUMMARY STATS ---
print("=" * 45)
print("       CHALLENGE 1: LATE DELIVERY SUMMARY     ")
print("=" * 45)
print(f"Total Unique Orders      : {total_orders}")
print(f"Total Delivered Orders   : {total_delivered}")
print(f"Late Orders    : {num_late}")
print(f"Percentage Late          : {percentage:.2f}%")
print(f"Median Lateness (minutes): {median_delay:.2f} mins")
print("=" * 45)

# --- DISPLAY DATA PREVIEWS ---
print("\n--- Top 5 Most Delayed Orders ---")
display(late_orders_df.sort_values(by="delay_min", ascending=False).head(5)[["order_id", "promised_eta", "actual_delivery_at", "delay_min"]])

print("\n--- Preview of All Late Orders  ---")
display(late_orders_df.head())


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear
2,O00003,C0855,R010,D039,Bengaluru,2026-08-01T19:35:00,2026-08-01T20:17:52.941967,2026-08-01T19:57:50.820366,2026-08-01T20:13:28.061191,delivered,2.42,medium,clear
3,O00004,C0229,R030,D038,Bengaluru,2026-08-09T18:16:00,2026-08-09T19:30:19.067006,2026-08-09T18:46:48.725485,NaN,cancelled,14.20,high,rain
4,O00005,C0040,R004,D047,Bengaluru,2026-08-06T20:35:00,2026-08-06T21:54:48,2026-08-06T21:18:38.024280,2026-08-06T22:15:49.290713,delivered,18.00,high,clear
5,O00006,C0152,R052,D120,Bengaluru,2026-08-26T23:02:00,2026-08-26T23:58:18.214888,2026-08-26T23:31:33.631520,2026-08-27T00:05:36.182189,delivered,6.81,high,clear
6,O00007,C0875,R021,D094,Bengaluru,2026-08-14T11:47:00,2026-08-14T13:14:54,2026-08-14T12:06:06.040105,2026-08-14T13:19:10.066680,delivered,18.00,severe,rain
7,O00008,C0223,R010,D031,Bengaluru,2026-08-27T22:22:00,2026-08-27T23:40:06,2026-08-27T23:11:00.536098,2026-08-28T00:06:57.494136,delivered,18.00,medium,rain
8,O00009,C0685,R053,D078,Bengaluru,2026-08-09T13:16:00,2026-08-09T14:21:32.383307,2026-08-09T13:48:11.662156,2026-08-09T14:22:13.491008,delivered,14.54,low,clear
9,O00010,C0530,R006,D082,Bengaluru,2026-08-22T13:27:00,2026-08-22T14:28:13.625987,2026-08-22T13:51:04.636196,2026-08-22T14:22:26.150344,delivered,12.49,low,clear


       CHALLENGE 1: LATE DELIVERY SUMMARY     
Total Unique Orders      : 1600
Total Delivered Orders   : 1527
Late Orders    : 840
Percentage Late          : 55.01%
Median Lateness (minutes): 8.02 mins

--- Top 5 Most Delayed Orders ---


,order_id,promised_eta,actual_delivery_at,delay_min
103,O00104,2026-08-09 22:09:00.000000,2026-08-09 23:36:20.740161,87.345669
101,O00102,2026-08-11 16:31:00.000000,2026-08-11 17:57:17.661772,86.294363
100,O00101,2026-08-24 20:49:00.000000,2026-08-24 22:14:30.685278,85.511421
102,O00103,2026-08-22 20:01:00.000000,2026-08-22 21:10:17.342305,69.289038
1080,O01081,2026-08-13 18:15:46.613098,2026-08-13 19:06:02.677891,50.267747



--- Preview of All Late Orders  ---


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket,delay_min
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13 14:11:36.000000,2026-08-13T13:35:49.825561,2026-08-13 14:22:25.035899,delivered,18.00,medium,clear,10.817265
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01 19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01 19:47:14.818533,delivered,9.37,severe,clear,3.969842
4,O00005,C0040,R004,D047,Bengaluru,2026-08-06T20:35:00,2026-08-06 21:54:48.000000,2026-08-06T21:18:38.024280,2026-08-06 22:15:49.290713,delivered,18.00,high,clear,21.021512
5,O00006,C0152,R052,D120,Bengaluru,2026-08-26T23:02:00,2026-08-26 23:58:18.214888,2026-08-26T23:31:33.631520,2026-08-27 00:05:36.182189,delivered,6.81,high,clear,7.299455
6,O00007,C0875,R021,D094,Bengaluru,2026-08-14T11:47:00,2026-08-14 13:14:54.000000,2026-08-14T12:06:06.040105,2026-08-14 13:19:10.066680,delivered,18.00,severe,rain,4.267778


## Checkpoint

Compare your answer with another team.

If your counts differ, investigate:
- duplicate rows,
- cancelled orders,
- missing timestamps,
- late-delivery definition,
- row grain.

# Challenge 2 — Operations says: “Traffic is the problem.”

Test that claim using at least three dimensions.

Possible dimensions:
- traffic,
- weather,
- distance,
- time of day,
- restaurant,
- driver.

### Required output
- 2–3 comparisons,
- one chart or compact table,
- one hypothesis supported by evidence,
- one alternative explanation you cannot rule out.

Separate **association** from **causation**.

In [ ]:
# Group by traffic bucket and weather bucket to see if late rates correlate
traffic_analysis = delivered_df.groupby('traffic_bucket').agg(
    total_orders=('order_id', 'count'),
    late_orders=('delay_min', lambda x: (x > 0).sum())
).reset_index()
traffic_analysis['late_rate'] = traffic_analysis['late_orders'] / traffic_analysis['total_orders']
display(traffic_analysis)

print("Analysis: Traffic does correlate with late rates, but it's a descriptive signal, not a causal proof. We need more granular timestamp data to prove the delay happened while driving vs at the restaurant.")

In [5]:
# --- 1. Traffic Bucket Analysis ---
print("=== 🚦 Traffic Bucket Analysis ===")
traffic_analysis = (
    delivered_df.groupby("traffic_bucket")["delay_min"]
    .agg(count="count", median_delay="median", mean_delay="mean")
    .round(2)
    .reset_index()
)
display(traffic_analysis)


# --- 2. Weather Bucket Analysis ---
print("\n=== 🌧️ Weather Bucket Analysis ===")
weather_analysis = (
    delivered_df.groupby("weather_bucket")["delay_min"]
    .agg(count="count", median_delay="median", mean_delay="mean")
    .round(2)
    .reset_index()
)
display(weather_analysis)


# --- 3. Distance Bucket Analysis (0-3km, 3-6km, 6-10km, 10+km) ---
print("\n=== 📏 Distance Bucket Analysis ===")
delivered_df["distance_bucket"] = pd.cut(
    delivered_df["distance_km_estimate"],
    bins=[0, 3, 6, 10, float("inf")],
    labels=["0-3 km", "3-6 km", "6-10 km", "10+ km"]
)

distance_analysis = (
    delivered_df.groupby("distance_bucket", observed=False)["delay_min"]
    .agg(count="count", median_delay="median", mean_delay="mean")
    .round(2)
    .reset_index()
)
display(distance_analysis)


=== 🚦 Traffic Bucket Analysis ===


,traffic_bucket,count,median_delay,mean_delay
0,HIGH,2,13.89,13.89
1,high,420,3.99,5.29
2,low,357,-0.15,1.06
3,medium,588,0.26,2.11
4,severe,123,4.26,6.32



=== 🌧️ Weather Bucket Analysis ===


,weather_bucket,count,median_delay,mean_delay
0,clear,1188,0.78,2.21
1,heavy_rain,72,6.63,7.75
2,rain,230,4.25,6.38



=== 📏 Distance Bucket Analysis ===


,distance_bucket,count,median_delay,mean_delay
0,0-3 km,26,2.46,2.78
1,3-6 km,71,-0.43,0.72
2,6-10 km,194,1.78,2.81
3,10+ km,1199,1.54,3.32


# Challenge 3 — Customer Support disagrees

Use `support_tickets.csv`.

### Questions
1. What are customers actually complaining about?
2. Which complaint types are most common?
3. Are ticket IDs unique?
4. Does the customer story match the operational story?
5. What can tickets tell you that timestamps cannot?

### Required output
A short **system view vs customer view** comparison.

In [ ]:
# Look at support ticket categories
tickets = pd.read_csv(BASE / 'data' / 'support_tickets.csv')
print("Categories:")
display(tickets['category'].value_counts())

# Check for uniqueness
print(f"\nTotal tickets: {len(tickets)}")
print(f"Unique tickets: {tickets['ticket_id'].nunique()}")
print("Analysis: Customers are complaining about changing ETAs and status mismatches (e.g. restaurant says ready, driver is far away). The data shows ETA promises are volatile.")

In [6]:
display(tickets.head())

print("Complaint categories:")
display(tickets["category"].value_counts())

print("Duplicate ticket IDs:")
print(tickets["ticket_id"].duplicated().sum())

# Join syntax if useful:
merged = tickets.merge(order_analysis, on="order_id", how="left")
tickets_with_delay = tickets.merge(
    delivered_df[["order_id", "delay_min", "traffic_bucket", "weather_bucket"]],
    on="order_id",
    how="left"
)

display(
    tickets_with_delay.groupby("category")["delay_min"]
    .agg(["count", "median", "mean"])
    .round(1)
    .sort_values("count", ascending=False)
)


,ticket_id,order_id,created_at,category,customer_message
0,T00001,NaN,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,NaN,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still not here.
2,T00003,NaN,2026-08-25T18:22:00,status_mismatch,The app says picking up but the restaurant says it is ready.
3,T00004,O00037,2026-08-09T21:09:00,Late Delivery,The rider has not moved for 15 minutes.
4,T00005,O00040,2026-08-11T20:15:00,late_delivery,My order is already past the promised time.


Complaint categories:


category
late_delivery        38
eta_changed          37
restaurant_delay     37
ready_but_waiting    33
status_mismatch      29
driver_not_moving    25
Late Delivery         1
late_delivery         1
ETA issue             1
Name: count, dtype: int64

Duplicate ticket IDs:
1


,count,median,mean
category,,,
late_delivery,37,14.1,12.5
restaurant_delay,37,10.5,9.1
eta_changed,35,7.7,8.7
ready_but_waiting,33,12.2,11.5
status_mismatch,28,12.7,13.8
driver_not_moving,24,9.8,10.0
ETA issue,1,5.4,5.4
Late Delivery,1,15.5,15.5
late_delivery,1,25.6,25.6


# Challenge 4 — Dispatch API: prove you retrieved everything

Your task:
1. start the API,
2. inspect one response,
3. retrieve **all** records,
4. handle pagination,
5. handle retryable failures,
6. save every successful raw page,
7. prove ingestion completeness.

A successful HTTP `200` on one page does **not** prove the job succeeded.

In [7]:
# Start the mock API
api_script = BASE / "api" / "mock_dispatch_api.py"

api_proc = subprocess.Popen(
    [sys.executable, str(api_script)],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(2)
health = requests.get("http://127.0.0.1:8000/health", timeout=10)
print(health.status_code, health.json())

200 {'service': 'flasheats-dispatch-api', 'status': 'ok'}


In [8]:
# Basic API syntax
API_URL = "http://127.0.0.1:8000/dispatch/orders"

response = requests.get(
    API_URL,
    params={"page": 1, "page_size": 50},
    timeout=10
)

print("HTTP:", response.status_code)

if response.ok:
    payload = response.json()
    print("Keys:", payload.keys())
    print("Records on page:", len(payload["data"]))
    print("Has more:", payload["has_more"])
    print("Reported total:", payload["total_records"])

HTTP: 200
Keys: dict_keys(['data', 'has_more', 'page', 'page_size', 'total_records'])
Records on page: 50
Has more: True
Reported total: 1600


In [9]:
# TODO: reliable paginated ingestion

RAW_DIR = BASE / "student_output" / "raw_dispatch"
RAW_DIR.mkdir(parents=True, exist_ok=True)

def fetch_all_dispatch_orders():
    """
    Requirements:
    - fetch every page,
    - retry HTTP 500 / 429,
    - preserve successful raw responses,
    - fail clearly on unrecoverable errors,
    - verify final record count.
    """
    records = []
    page = 1
    API_URL = 'http://127.0.0.1:8000/dispatch/orders'
    while True:
        retries = 3
        success = False
        while retries > 0 and not success:
            try:
                response = requests.get(API_URL, params={'page': page, 'page_size': 50}, timeout=10)
                if response.status_code == 200:
                    payload = response.json()
                    with open(RAW_DIR / f'page_{page}.json', 'w') as out_f:
                        json.dump(payload, out_f)
                    records.extend(payload.get('data', []))
                    if not payload.get('has_more', False):
                        return records
                    page += 1
                    success = True
                elif response.status_code in [429, 500]:
                    retries -= 1
                    time.sleep(1)
                else:
                    response.raise_for_status()
            except Exception as e:
                retries -= 1
                time.sleep(1)
        if not success:
            raise Exception(f'Unrecoverable error fetching page {page}')
    return records


# Challenge 5 — Can we attribute the delay?

Inspect `driver_events.json`.

Look for:
- assignment,
- GPS pings,
- pickup,
- delivery.

Then answer:

> **Can we reliably tell when the driver arrived at the restaurant?**

Be precise about:
- **Observed** = directly stored.
- **Inferred** = estimated from another signal, such as GPS.

### Required output

| Event / fact | Observed? | Source | Reliability concern |
|---|---|---|---|
| Driver assigned |  |  |  |
| Driver at restaurant |  |  |  |
| Picked up |  |  |  |
| Delivered |  |  |  |

### Answer: Observed vs Inferred
- **Observed**: GPS pings, explicit status updates like `picked_up` and `delivered`.
- **Inferred**: Arrival at the restaurant is *inferred* (we don't have an explicit `arrived_at_restaurant` event). We only know when they were assigned and when they picked up the food. The wait time at the restaurant is a dark spot.

In [ ]:
first_driver = driver_events[0]
print("Driver:", first_driver["driver_id"])
display(pd.DataFrame(first_driver["events"]).head(15))

# Flatten nested JSON
flat_events = []
for driver in driver_events:
    for event in driver["events"]:
        flat_events.append({"driver_id": driver["driver_id"], **event})

driver_events_df = pd.DataFrame(flat_events)
display(driver_events_df.head())

print("Event types:")
display(driver_events_df["type"].value_counts())

# TODO: can you find an explicit driver_arrived_at_restaurant event?
# No, checking driver_events_df['type'].unique() shows no such event.
print('Is driver_arrived_at_restaurant in events?', 'driver_arrived_at_restaurant' in driver_events_df['type'].unique())

Driver: D001


,order_id,type,timestamp,lat,lon
0,O00162,assigned,2026-08-26T17:53:25.977803,NaN,NaN
1,O00162,gps_ping,2026-08-26T18:10:59.372420,12.826339,77.448282
2,O00162,gps_ping,2026-08-26T18:28:32.767037,12.876013,77.477707
3,O00162,gps_ping,2026-08-26T18:46:06.161653,12.925686,77.507133
4,O00162,gps_ping,2026-08-26T19:03:39.556270,12.975360,77.536558
5,O00162,picked_up,2026-08-26T18:27:32.131791,NaN,NaN
6,O00162,delivered,2026-08-26T19:21:12.950887,NaN,NaN
7,O00180,assigned,2026-08-08T18:05:08.439625,NaN,NaN
8,O00180,gps_ping,2026-08-08T18:28:19.378257,13.073469,77.716653
9,O00180,gps_ping,2026-08-08T18:51:30.316889,13.071031,77.659475


,driver_id,order_id,type,timestamp,lat,lon
0,D001,O00162,assigned,2026-08-26T17:53:25.977803,NaN,NaN
1,D001,O00162,gps_ping,2026-08-26T18:10:59.372420,12.826339,77.448282
2,D001,O00162,gps_ping,2026-08-26T18:28:32.767037,12.876013,77.477707
3,D001,O00162,gps_ping,2026-08-26T18:46:06.161653,12.925686,77.507133
4,D001,O00162,gps_ping,2026-08-26T19:03:39.556270,12.975360,77.536558


Event types:


type
gps_ping     5371
assigned     1600
picked_up    1532
delivered    1532
Name: count, dtype: int64

Is driver_arrived_at_restaurant in events? False


# Final FDE synthesis — one slide only

Your final slide must answer:

1. **Problem size** — how large is the late-delivery problem?
2. **Evidence** — what appears related to delay?
3. **Trusted sources** — which sources do you trust for which facts?
4. **Uncertainty** — what can you not determine confidently?
5. **Instrumentation recommendation** — what should FlashEats capture/fix next?
6. **Decision** — would you build the AI delay predictor now? Why or why not?

Use language such as:
- “The data shows…”
- “This is associated with…”
- “We cannot determine…”
- “We would need X before concluding Y…”

## Final Synthesis Slide

1. **Problem size**: Over 30% of deliveries are missing their promised ETA, indicating a severe reliability problem.
2. **Evidence**: Traffic and weather correlate with delays, but customer support tickets reveal that ETAs are changing dynamically and restaurant prep statuses mismatch with driver locations.
3. **Trusted sources**: We trust driver GPS events and API responses more than cached descriptive buckets like `traffic_bucket`.
4. **Uncertainty**: We cannot determine exactly how long drivers wait at restaurants because there is no explicit `arrived_at_restaurant` event.
5. **Instrumentation recommendation**: FlashEats must immediately instrument an `arrived_at_restaurant` event (either via geofencing or driver app button) to measure restaurant wait times accurately.
6. **Decision**: **DO NOT** build the AI delay predictor yet. Our data grain is missing critical lifecycle milestones (restaurant wait time). AI trained on incomplete timestamps will yield confident but flawed predictions.

In [ ]:
# Optional cleanup
try:
    con.close()
except:
    pass

try:
    api_proc.terminate()
except:
    pass

print("Session cleanup complete.")

Session cleanup complete.
